# Prompt Firewall — Train on Colab (GPU)

1. **Runtime → Change runtime type → T4 GPU**
2. Clone repo, install deps, load training data, run `train_classifier.py`
3. Download `models/classifier-v1/` and `reports/` when done

Data is not in git (DVC/gitignored). Use **Option A** (Google Drive zip) or **Option B** (re-ingest).

In [ ]:
# Clone repo (update branch if needed)
!git clone https://github.com/akshatakumble/prompt-firewall.git
%cd prompt-firewall

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Install pinned deps (GPU torch already on Colab; install the rest)
!pip install -q -r requirements-dev.txt

## Option A — Load splits from Google Drive (recommended)

On your PC, zip this folder and upload to Drive:
`data/curated/v1.0/splits/` → `splits.zip`

In [ ]:
from google.colab import drive
import zipfile
from pathlib import Path

drive.mount('/content/drive')

# CHANGE THIS to where you uploaded splits.zip on Drive
ZIP_PATH = '/content/drive/MyDrive/splits.zip'

out = Path('data/curated/v1.0/splits')
out.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(out)
print('Files:', list(out.glob('*.parquet')))

# Manifest for MLflow traceability (optional)
manifest_src = Path('/content/drive/MyDrive/manifest.json')  # optional upload
if manifest_src.exists():
    Path('data/registry/v1.0').mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy(manifest_src, 'data/registry/v1.0/manifest.json')

## Option B — Re-ingest on Colab (slow; needs HF_TOKEN)

Skip Option A cells and run below if you have WildJailbreak raw data or HF access.

In [ ]:
# import os
# os.environ['HF_TOKEN'] = 'hf_...'  # from huggingface.co/settings/tokens
# !python scripts/ensure_pipeline_dirs.py
# !python pipelines/ingest_dataset.py --config config/app.yaml

In [ ]:
# Full training (~20-40 min on T4) — hyperparam grid + DistilBERT
!python pipelines/train_classifier.py --config config/app.yaml --epochs 2

In [ ]:
# Quick smoke test (~5 min) — uncomment instead of full train above
# !python pipelines/train_classifier.py --config config/app.yaml --quick --epochs 1 --max-train 2000

In [ ]:
# Zip artifacts back to Drive
!zip -r classifier_artifacts.zip models/classifier-v1 reports mlruns
!cp classifier_artifacts.zip /content/drive/MyDrive/

## After Colab

On your PC:
1. Download `classifier_artifacts.zip` from Drive
2. Extract `models/classifier-v1/` into your local repo
3. Run evaluate: `python pipelines/evaluate_firewall.py --benchmark salad-data`